# Week 5: ablation checkpoints on the test sets
Every run's final (step 250) 0.5B LoRA adapter, attached as one dataset, scored on GSM8K test and SVAMP with the week 1 harness settings (vLLM, fp16, greedy, 1,024 tokens). The adapters are split across the two GPUs; each GPU scores its adapters one after another.

In [ ]:
import os, subprocess
REPO, SRC = "https://github.com/Vansh8010-boop/grpo-reasoning.git", "/tmp/grpo-reasoning"
if os.path.isdir(SRC):
    subprocess.run(["git", "-C", SRC, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q vllm "math-verify[antlr4_13_2]" 2>&1 | tail -3
# vLLM brings its own torch build; Kaggle's preinstalled torchaudio no longer matches it
# and transformers imports it on startup, so remove it (no audio needed here).
!pip uninstall -y -q torchaudio
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__)"

In [ ]:
!python -m pytest -q tests/ 2>&1 | tail -3

In [ ]:
import glob
OUT = "/kaggle/working"
adapters = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
names = [os.path.basename(os.path.dirname(a)) for a in adapters]  # <run>/ckpt-0250/adapter_config.json
print(len(adapters), "adapters:", names)
assert len(set(names)) == len(names), names
env = {**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
shards = {"0": list(zip(names, adapters))[0::2], "1": list(zip(names, adapters))[1::2]}
script = {gpu: " && ".join(
    f"python -m eval.harness --model Qwen/Qwen2.5-0.5B-Instruct --dtype float16 --datasets gsm8k,svamp "
    f"--adapter {a} --out {OUT}/results/{n} > {OUT}/{n}.log 2>&1" for n, a in jobs) for gpu, jobs in shards.items()}
procs = {gpu: subprocess.Popen(["bash", "-c", cmd], env={**env, "CUDA_VISIBLE_DEVICES": gpu}) for gpu, cmd in script.items()}
for gpu, p in procs.items():
    print("GPU", gpu, "exit code", p.wait())
jobs = {n: None for n in names}
for n in names:
    print(n, "".join(l for l in open(f"{OUT}/{n}.log") if l.startswith("[") and "accuracy" in l or "Error" in l))

In [ ]:
import json
rows = []
for name in jobs:
    path = f"{OUT}/results/{name}/summary.json"
    if not os.path.exists(path):
        print("missing", path); continue
    for ds, s in json.load(open(path))["datasets"].items():
        rows.append(f"{name:10s} {ds:8s} acc {s['accuracy']*100:5.1f}%  boxed {s['boxed_rate']*100:5.1f}%  "
                    f"trunc {s['truncated_rate']*100:4.1f}%  tokens {s['avg_tokens']:4.0f}  {s['seconds']}s")
print("\n".join(rows))
os.makedirs(f"{OUT}/failures", exist_ok=True)
for name in jobs:
    for ds in ("gsm8k", "math500", "svamp"):
        src = f"{OUT}/results/{name}/{ds}.jsonl"
        if os.path.exists(src):
            subprocess.run(["python", "-m", "eval.failures", src, "-n", "20"],
                           stdout=open(f"{OUT}/failures/{name}-{ds}.md", "w"), check=True)